# 1단계 PLATOON1C — '이득이 확실할 때만 예측 제어와 다르게 고르기': 문턱 t30

**무엇을 하나:** PT1은 결정의 40%를 예측 제어 TM4와 다르게 골라 TM4보다 1.03초 길었습니다. 이번에는 **같은 학생 3개**를 그대로 쓰고,
예측 이득이 문턱 이상일 때만 다르게 고릅니다. 새 수집·재학습은 없고 SUMO 평가만 합니다. 기준은 실행 전에 고정했습니다
(`results/explore/platoon1c_20261101/사전등록.md`).

**이 파일 = 문턱 τ = 3.0 (학습 자료에서 바꾸는 비율 약 5%).**

**이 파일 하나 = 런타임 하나입니다.** `PLATOON1C_t09`, `PLATOON1C_t20`, `PLATOON1C_t30` 세 파일을 하나씩 열어(런타임 유형 **CPU + 높은 RAM**)
각각 **런타임 → 모두 실행**을 누르세요. 동시에 돌리면 각 2~3시간, 하나씩 돌려도 됩니다.

순서: 묶음 검증 → 패키지 → 시험 → 작은 실행(3판, 약 5분) → 평가(252판, 백그라운드).
끊기면 다시 **모두 실행**하면 끝난 판은 건너뜁니다. 세 파일이 모두 끝나면 `PLATOON1C_judge`를 실행하세요.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, json

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/FutureTraffic")
ARCHIVE = DRIVE / "FutureTraffic_PLATOON1C.zip"
PLATOON0 = DRIVE / "explore_platoon0"          # 무리 도착 세계 M4·TM4·OM4·P1M4(읽기만)
PLATOON1 = DRIVE / "explore_platoon1"          # 학생 3개와 PT1 평가(읽기만)
PBASE = DRIVE / "explore_platoon_base"         # 무리 도착 세계 고정신호·P0(읽기만)
STUDENTS = PLATOON1 / "students"
OUT = DRIVE / "explore_platoon1c"
WORKSPACE = Path("/content/FutureTraffic_PLATOON1C")
MIN_CPUS = 8
for path in (ARCHIVE, PLATOON0 / "experiment.json", PLATOON0 / "runs" / "P" / "TM4", PLATOON1 / "eval" / "PT1" / "experiment.json",
             PBASE / "runs" / "P" / "FIXC", *(STUDENTS / f"seed_{s}" / "model.pt" for s in (42, 43, 44))):
    assert path.exists(), f"필요한 파일이 없습니다: {path}"
CPUS = os.cpu_count() or 1
CANDIDATE = "t30"      # 이 파일 = 문턱 τ = 3.0(바꾸지 마세요)
print("CPU 개수:", CPUS, "| 후보:", CANDIDATE, "| 결과 폴더:", OUT / "eval" / CANDIDATE)
assert CPUS >= MIN_CPUS, (f"CPU가 {CPUS}개입니다. 런타임 → 런타임 유형 변경에서 CPU와 '높은 RAM'을 고른 뒤 다시 실행하세요.")

In [ ]:
# ZIP 전체의 파일 목록·SHA-256을 확인한 뒤 검증된 전용 추출기를 사용합니다.
import hashlib, stat, types, zipfile
from pathlib import PurePosixPath

with zipfile.ZipFile(ARCHIVE) as archive:
    names = archive.namelist()
    assert len(names) == len(set(names)) and names.count("manifest.json") == 1, "ZIP 파일 목록 오류"
    package_manifest = json.loads(archive.read("manifest.json"))
    assert package_manifest.get("kind") == "WEIGHT1_PACKAGE_V1", "검증할 수 없는 ZIP 형식입니다"
    assert package_manifest.get("experiment") == "PLATOON1C_V1", "PLATOON1C 전용 ZIP이 아닙니다"
    assert package_manifest.get("source_models_included") is False, "모델·결과가 든 묶음은 쓰지 않습니다"
    expected = package_manifest["files"]
    assert expected and "manifest.json" not in expected and set(names) == set(expected) | {"manifest.json"}, "ZIP과 manifest 목록 불일치"
    for info in archive.infolist():
        name, path = info.filename, PurePosixPath(info.filename)
        assert path.parts and not path.is_absolute() and ".." not in path.parts and path.as_posix() == name, "잘못된 ZIP 경로"
        assert "\\" not in name and not info.is_dir() and not stat.S_ISLNK(info.external_attr >> 16), "잘못된 ZIP 파일 형식"
        if name != "manifest.json":
            assert hashlib.sha256(archive.read(name)).hexdigest() == expected[name], f"SHA-256 불일치: {name}"
    source_code = archive.read("src/explore_weight_package.py").decode("utf-8")

bootstrap = types.ModuleType("platoon1c_verified_package")
bootstrap.__file__ = str(WORKSPACE / "src/explore_weight_package.py")
exec(compile(source_code, bootstrap.__file__, "exec"), bootstrap.__dict__)
package_manifest = bootstrap.extract_verified(ARCHIVE, WORKSPACE)
print(f"검증·추출 완료: {len(package_manifest['files'])}개 파일 → {WORKSPACE} | 잠금 {package_manifest['lock_sha256'][:12]} "
      f"| PLATOON1C 설정 {package_manifest['platoon1c_config_sha256'][:12]} | 사전 등록 {package_manifest['prereg_sha256'][:12]}")

In [ ]:
import subprocess, importlib.metadata

os.environ["FOCUSED_RULESET"] = "pet1"
os.environ["FOCUSED_NORTH_EXT"] = "240"
# ARENA3 심판 기준선과 같은 평가 지문이 되도록 같은 패키지 버전을 씁니다.
EXPECTED = {"numpy": "2.1.3", "torch": "2.11.0+cpu", "stable-baselines3": "2.9.0",
            "sumo-rl": "1.4.5", "eclipse-sumo": "1.27.1"}

def installed_versions():
    versions = {}
    for name in EXPECTED:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return versions

before = installed_versions()
if before == EXPECTED:
    print("이전 실험과 패키지 버전이 같아 재설치를 건너뜁니다.")
else:
    print("이전 실험과 같은 패키지 버전을 준비합니다:", before)
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "numpy==2.1.3", "stable-baselines3==2.9.0", "eclipse-sumo==1.27.1"],
                   cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.11.0+cpu", "--index-url", "https://download.pytorch.org/whl/cpu"],
                   cwd=WORKSPACE, check=True)
after = installed_versions()
assert after == EXPECTED, f"패키지 버전이 이전 실험과 다릅니다(평가 지문이 달라져 기준선과 비교할 수 없어 멈춥니다): {after}"
print("실행 패키지 검증 통과:", after)

In [ ]:
sys.path.insert(0, str(WORKSPACE / "src"))
import arena3_colab as colab
from IPython.display import display, Markdown

for name in ("test_explore_platoon1c.py", "test_explore_platoon1.py"):
    colab.run_live(["-m", "unittest", "discover", "-s", "tests", "-p", name], WORKSPACE)
print("실행 전 검사(SUMO 없이) 통과")

In [ ]:
# 1) 작은 실행(약 5분): 첫 평가 상황 3판(난수 42 모델).
#    - 문턱을 아주 높이면 PLATOON0의 TM4와 같은가, 문턱을 0에 가깝게 두면 PLATOON1의 PT1과 같은가, 이 문턱은 PT1보다 덜 바꾸는가.
SMOKE = OUT / "smoke" / CANDIDATE
previous = colab.read_json(SMOKE / "smoke.json")
if previous and previous.get("ok"):
    print("이전 작은 실행이 통과해 건너뜁니다.")
else:
    colab.run_live(["src/explore_platoon1c.py", "smoke", "--candidate", CANDIDATE, "--students", str(STUDENTS), "--out", str(SMOKE),
                    "--work", "/content/pc_smoke", "--workers", "3", "--platoon0", str(PLATOON0), "--platoon1", str(PLATOON1)],
                   WORKSPACE, check=False)
s = colab.read_json(SMOKE / "smoke.json")
assert s and s.get("ok"), f"작은 실행 점검 실패 — 이 칸의 출력을 저에게 보내 주세요: {s and s.get('checks')}"
print("작은 실행 통과:", s["checks"])
print("이 상황에서 TM4와 다르게 고른 횟수:", s["deviations"])

In [ ]:
# 2) 평가: 모델 3개 × 84상황 = 252판(백그라운드). 셀을 멈춰도 계속 돕니다.
#    런타임이 끊기면 처음 네 칸 → 이 칸을 다시 실행하세요. 끝난 판은 건너뜁니다.
assert "colab" in globals(), "처음 네 칸(1~4)을 먼저 실행하세요. 런타임이 끊기면 변수와 작업 폴더가 사라집니다."
EVAL = OUT / "eval" / CANDIDATE
LOGS = OUT / "logs"


def describe(name):
    done = len(list((EVAL / "runs").glob(f"{CANDIDATE}_m*/*/cache.json")))
    failed = len(list((EVAL / "runs").glob(f"{CANDIDATE}_m*/*.failed.json")))
    return f"끝난 판 {done}/252, 실패 기록 {failed}"


pids = colab.launch({f"evaluate_{CANDIDATE}": ["src/explore_platoon1c.py", "evaluate", "--candidate", CANDIDATE, "--students", str(STUDENTS),
                                               "--out", str(EVAL), "--work", f"/content/pc_{CANDIDATE}", "--workers", str(CPUS)]},
                    WORKSPACE, LOGS)
code = colab.watch(pids, LOGS, 600, describe)
assert code == 0, "평가에서 오류가 났습니다. 위 로그를 저에게 보내 주세요(다시 실행하면 끝난 판은 건너뜁니다)."
print(describe(None), "— 세 파일이 모두 끝나면 PLATOON1C_judge를 실행하세요.")

In [ ]:
# 3) 세 문턱의 진행(언제든 다시 실행해 볼 수 있습니다).
for c in ("t09", "t20", "t30"):
    done = len(list((OUT / "eval" / c / "runs").glob(f"{c}_m*/*/cache.json")))
    print(c, f"{done}/252")